In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import napari
import load_assess_image as load_assess
import anndata as ad
import pandas as pd

In [16]:
main_dir = rf'D:\thu\HTAN\images\NST'
file_name= 'TNP-TMA-7-Scene-F01.ome.tif'
image_test = load_assess.AssessImage(main_dir, file_name)

In [17]:
# Open mask in NPZ form - faster to reload
mask = np.load(rf"D:\thu\HTAN\images\mask\NST\{file_name.split('.')[0]}.npz")
mask['nuclei']

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5731, 5728))

In [18]:
dna_round_1 = image_test.assess_single_image('dna', resolution_level=0)
dna_round_6 = image_test.assess_single_image('dna(6)', resolution_level=0)
dna_round_7 = image_test.assess_single_image('dna(7)', resolution_level=0)
pRB = image_test.assess_single_image('pRB', resolution_level=0)
ki67 = image_test.assess_single_image('Ki67', resolution_level=0)

In [ ]:
viewer = napari.Viewer()
viewer.add_image(dna_round_1, name='dna_round_1', colormap='gray', blending='additive')
viewer.add_image(dna_round_6, name='dna_round_6', colormap='gray', blending='additive')
viewer.add_image(dna_round_7, name='dna_round_7', colormap='gray', blending='additive')
viewer.add_image(pRB, name='pRB', colormap='gray', blending='additive')
viewer.add_image(ki67, name='ki67', colormap='gray', blending='additive')
viewer.add_labels(mask['nuclei'], name='nuclei_mask', opacity=0.5, blending='additive')

<Labels layer 'nuclei_mask' at 0x22c56ae5550>

: 

In [7]:
mip_cyto = np.load(rf"D:\thu\HTAN\images\LSP12021-C10\MIP_cyto\LSP12021-C10.npz")

In [8]:
mip_cyto.files

['cyto_mip']

In [12]:
df = pd.read_csv(rf"R:\Thu\From_2026\6. Other\DL_Summer_2026\output\LSP12021-C10_cells.csv")

In [14]:
df['label'] =df['label'].astype(object)

In [18]:
obs = df.select_dtypes(include=['object', 'category']).copy()
X_df = df.select_dtypes(include=['number'])   # only numeric feature columns

adata = ad.AnnData(
    X=X_df.values,
    obs=obs,
    var=pd.DataFrame(index=X_df.columns)
)


c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [19]:
qc_tissues = pd.read_csv(rf"D:\thu\HTAN\images\QC_Fold_Lost\NST\LSP12021-C10\LSP12021-C10_flagged_bounds.csv")

In [63]:
qc_tissues[qc_tissues['round_dna'] =='dna(10)']

,filename,round_dna,tile_type,row,col,x0,y0,x1,y1


In [48]:
columns = ['label',
 'area',
 'centroid-0',
 'centroid-1',
 'equivalent_diameter_area',
 'solidity',
 'eccentricity',
 'axis_major_length',
 'axis_minor_length',
 'perimeter',
 'dna_mean',
 'dna_median',
 'dna_std',
 'control488nm_mean',
 'control488nm_median',
 'control488nm_std',
 'control555nm_mean',
 'control555nm_median',
 'control555nm_std',
 'control647nm_mean',
 'control647nm_median',
 'control647nm_std',
 'dna(8)_mean',
 'dna(8)_median',
 'dna(8)_std',
 'CD57_mean',
 'CD57_median',
 'CD57_std',
 'PDL1-28-8_mean',
 'PDL1-28-8_median',
 'PDL1-28-8_std',
 'CD3d_mean',
 'CD3d_median',
 'CD3d_std',]

In [49]:
df_test = df[columns]

In [50]:
qc_tissues_test = qc_tissues[qc_tissues['round_dna'] =='dna(8)']

In [51]:
df_test.columns


Index(['label', 'area', 'centroid-0', 'centroid-1', 'equivalent_diameter_area',
       'solidity', 'eccentricity', 'axis_major_length', 'axis_minor_length',
       'perimeter', 'dna_mean', 'dna_median', 'dna_std', 'control488nm_mean',
       'control488nm_median', 'control488nm_std', 'control555nm_mean',
       'control555nm_median', 'control555nm_std', 'control647nm_mean',
       'control647nm_median', 'control647nm_std', 'dna(8)_mean',
       'dna(8)_median', 'dna(8)_std', 'CD57_mean', 'CD57_median', 'CD57_std',
       'PDL1-28-8_mean', 'PDL1-28-8_median', 'PDL1-28-8_std', 'CD3d_mean',
       'CD3d_median', 'CD3d_std'],
      dtype='object')

In [ ]:
import re

def get_dna_round_order(columns):
    """
    Ordered list of DNA round names as they appear in df's columns
    (e.g. ['dna', 'dna(2)', 'dna(3)', ...]), matching the imaging-cycle order.
    """
    pattern = re.compile(r'^dna(\(\d+\))?_mean$')
    return [col[:-len('_mean')] for col in columns if pattern.match(col)]


def exclude_flagged_cells(df, flagged_bounds_df, x_col='centroid-1', y_col='centroid-0'):
    """
    For each round_dna present in flagged_bounds_df, find cells whose centroid
    falls inside any flagged tile for that round, and set NaN across the
    imaging-cycle column block from that round's _mean through the NEXT
    round's _std (inclusive) -- i.e. the DNA channel + markers stained in
    that same physical cycle, since a fold/lost-tissue artifact detected at
    a given DNA round corrupts everything imaged in that cycle.

    Assumes centroid-0 = row = y, centroid-1 = col = x (skimage regionprops
    convention), matching how x0/y0/x1/y1 were built by
    QC_Fold_Lost.get_flagged_tile_bounds().
    """
    df = df.copy()
    dna_round_order = get_dna_round_order(df.columns)

    for round_dna, group in flagged_bounds_df.groupby('round_dna'):
        if round_dna not in dna_round_order:
            print(f"Skipping {round_dna}: not found in df columns")
            continue

        start_col = f'{round_dna}_mean'
        idx = dna_round_order.index(round_dna)
        end_col = f'{dna_round_order[idx + 1]}_std' if idx + 1 < len(dna_round_order) else df.columns[-1]

        x = df[x_col].to_numpy()
        y = df[y_col].to_numpy()
        in_any_flagged_tile = np.zeros(len(df), dtype=bool)
        for _, tile in group.iterrows():
            in_any_flagged_tile |= (
                (x >= tile['x0']) & (x < tile['x1']) &
                (y >= tile['y0']) & (y < tile['y1'])
            )

        df.loc[in_any_flagged_tile, start_col:end_col] = np.nan
        print(f"{round_dna}: {in_any_flagged_tile.sum()} cells NaN'd across {start_col}..{end_col}")

    return df


In [57]:
sdata_df_test = exclude_flagged_cells(df, qc_tissues, x_col='centroid-1', y_col='centroid-0')

dna(2): 281 cells NaN'd across dna(2)_mean..dna(3)_std
dna(3): 420 cells NaN'd across dna(3)_mean..dna(4)_std
dna(4): 683 cells NaN'd across dna(4)_mean..dna(5)_std
dna(5): 675 cells NaN'd across dna(5)_mean..dna(6)_std
dna(6): 800 cells NaN'd across dna(6)_mean..dna(7)_std
dna(7): 1679 cells NaN'd across dna(7)_mean..dna(8)_std
dna(8): 1776 cells NaN'd across dna(8)_mean..dna(9)_std
dna(9): 1266 cells NaN'd across dna(9)_mean..dna(10)_std


In [64]:
import xarray as xr
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, Labels2DModel, PointsModel, TableModel

# ---- AnnData from the QC-cleaned cell table ----
sdata_df_test['label'] = sdata_df_test['label'].astype(int)   # match mask's int label IDs
sdata_df_test['region'] = 'segmentation'

non_feature_cols = ['label', 'region', 'centroid-0', 'centroid-1', 'area',
                    'equivalent_diameter_area', 'solidity', 'eccentricity',
                    'axis_major_length', 'axis_minor_length', 'perimeter']
feature_cols = [c for c in sdata_df_test.columns if c not in non_feature_cols]

obs = sdata_df_test[['label', 'region']].copy()
X_df = sdata_df_test[feature_cols]

adata_qc = ad.AnnData(
    X=X_df.values,
    obs=obs,
    var=pd.DataFrame(index=X_df.columns),
)

table_element = TableModel.parse(
    adata_qc, region='segmentation', region_key='region', instance_key='label', overwrite_metadata=True
)

# ---- Images ----
dapi_element = Image2DModel.parse(xr.DataArray([dna_round_1], dims=('c', 'y', 'x')))
cyto_element = Image2DModel.parse(xr.DataArray([mip_cyto['cyto_mip']], dims=('c', 'y', 'x')))

# ---- Points (cell centroids: x, y order for PointsModel) ----
points_element = PointsModel.parse(sdata_df_test[['centroid-1', 'centroid-0']].to_numpy())

# ---- Labels (segmentation mask) ----
labels_element = Labels2DModel.parse(xr.DataArray(mask['cells'], dims=('y', 'x')))

# ---- Diagnostic: check label overlap between table and mask, same check as sdata_test.py ----
mask_ids = set(np.unique(mask['cells'])) - {0}
table_label_ids = set(sdata_df_test['label'].unique())
print(f"Cells in table: {len(table_label_ids)}, unique mask IDs: {len(mask_ids)}")
print(f"Matching: {len(mask_ids & table_label_ids)}, "
      f"in table not in mask: {len(table_label_ids - mask_ids)}, "
      f"in mask not in table: {len(mask_ids - table_label_ids)}")

sdata = SpatialData(
    images={"DAPI": dapi_element, "cyto_mip": cyto_element},
    labels={"segmentation": labels_element},
    points={"centroids": points_element},
    tables={"expression": table_element},
)
print(sdata)


c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\site-packages\spatialdata\models\models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


Cells in table: 11060, unique mask IDs: 11060
Matching: 11060, in table not in mask: 0, in mask not in table: 0
SpatialData object
├── Images
│     ├── 'DAPI': DataArray[cyx] (1, 2726, 2840)
│     └── 'cyto_mip': DataArray[cyx] (1, 2726, 2840)
├── Labels
│     └── 'segmentation': DataArray[yx] (2726, 2840)
├── Points
│     └── 'centroids': DataFrame with shape: (<Delayed>, 2) (2D points)
└── Tables
      └── 'expression': AnnData (11060, 120)
with coordinate systems:
    ▸ 'global', with elements:
        DAPI (Images), cyto_mip (Images), segmentation (Labels), centroids (Points)


In [61]:
import xarray as xr
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, Labels2DModel, PointsModel, TableModel

# ---- AnnData from the QC-cleaned cell table ----
df['label'] = df['label'].astype(int)   # match mask's int label IDs
df['region'] = 'segmentation'

non_feature_cols = ['label', 'region', 'centroid-0', 'centroid-1', 'area',
                    'equivalent_diameter_area', 'solidity', 'eccentricity',
                    'axis_major_length', 'axis_minor_length', 'perimeter']
feature_cols = [c for c in df.columns if c not in non_feature_cols]

obs = df[['label', 'region']].copy()
X_df = df[feature_cols]

adata_qc = ad.AnnData(
    X=X_df.values,
    obs=obs,
    var=pd.DataFrame(index=X_df.columns),
)

table_element = TableModel.parse(
    adata_qc, region='segmentation', region_key='region', instance_key='label', overwrite_metadata=True
)

# ---- Images ----
dapi_element = Image2DModel.parse(xr.DataArray([dna_round_1], dims=('c', 'y', 'x')))
cyto_element = Image2DModel.parse(xr.DataArray([mip_cyto['cyto_mip']], dims=('c', 'y', 'x')))

# ---- Points (cell centroids: x, y order for PointsModel) ----
points_element = PointsModel.parse(df[['centroid-1', 'centroid-0']].to_numpy())

# ---- Labels (segmentation mask) ----
labels_element = Labels2DModel.parse(xr.DataArray(mask['cells'], dims=('y', 'x')))

# ---- Diagnostic: check label overlap between table and mask, same check as sdata_test.py ----
mask_ids = set(np.unique(mask['cells'])) - {0}
table_label_ids = set(df['label'].unique())
print(f"Cells in table: {len(table_label_ids)}, unique mask IDs: {len(mask_ids)}")
print(f"Matching: {len(mask_ids & table_label_ids)}, "
      f"in table not in mask: {len(table_label_ids - mask_ids)}, "
      f"in mask not in table: {len(mask_ids - table_label_ids)}")

sdata = SpatialData(
    images={"DAPI": dapi_element, "cyto_mip": cyto_element},
    labels={"segmentation": labels_element},
    points={"centroids": points_element},
    tables={"expression": table_element},
)
print(sdata)


c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\site-packages\spatialdata\models\models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


Cells in table: 11060, unique mask IDs: 11060
Matching: 11060, in table not in mask: 0, in mask not in table: 0
SpatialData object
├── Images
│     ├── 'DAPI': DataArray[cyx] (1, 2726, 2840)
│     └── 'cyto_mip': DataArray[cyx] (1, 2726, 2840)
├── Labels
│     └── 'segmentation': DataArray[yx] (2726, 2840)
├── Points
│     └── 'centroids': DataFrame with shape: (<Delayed>, 2) (2D points)
└── Tables
      └── 'expression': AnnData (11060, 120)
with coordinate systems:
    ▸ 'global', with elements:
        DAPI (Images), cyto_mip (Images), segmentation (Labels), centroids (Points)


In [59]:
from napari_spatialdata import Interactive

In [ ]:
interactive = Interactive(sdata)
interactive.run()

2026-07-15 15:22:07.401 | WARNING  | napari_spatialdata._viewer:__init__:56 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


2026-07-15 15:22:18.257 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:18.366 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
c:\Users\anp525\AppData\Local\miniconda3\envs\spatialdata-env\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-15 15:22:47.319 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:47.329 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:47.332 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:49.303 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:49.313 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-07-15 15:22:49.431 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
